# WLASL Production Architecture: Real-Time English Word Recognition (BiLSTM)

### Enhancements
- **78-Dimensional Feature Engineering**: Uses 63 wrist-relative coordinates and 15 joint angles for robust pose-invariance.
- **Sliding Window Inference**: Maintains a 30-frame rolling window of engineered features for the BiLSTM model.
- **Threaded Camera**: Drop-queue design to ensure zero-latency frame fetching.
- **Compiled Inference (`@tf.function`)**: Graph-mode execution for fast prediction.
- **Stabilization Tracker**: Commit-once-then-wait logic to prevent flickering.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
import threading
import queue
from collections import deque
import os
import pandas as pd

print("✓ Libraries imported successfully.")

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠ GPU config error: {e}")
else:
    print("ℹ No GPU detected, using CPU")

In [ ]:
MODEL_PATH = "asl_word_lstm_model_best.h5"
CLASSES_CSV = "asl_word_classes.csv"

# Production Params
SEQUENCE_LENGTH            = 30     # BiLSTM window size
STABILIZATION_WINDOW_SIZE  = 15     # rolling-buffer length for predictions
STABILIZATION_THRESHOLD    = 11     # 11/15 majority required
MIN_CONFIDENCE             = 0.75   # predictions below this are discarded
HOLD_COOLDOWN_SECONDS      = 1.2    # lock-out after a commit

MP_DETECTION_CONFIDENCE    = 0.70
MP_TRACKING_CONFIDENCE     = 0.65

DISPLAY_WIDTH  = 1280
DISPLAY_HEIGHT = 720

# Load vocabulary
if os.path.exists(CLASSES_CSV):
    df = pd.read_csv(CLASSES_CSV)
    # create mapping from index to string label (word_id is string or int)
    CLASS_LABELS = [str(word) for word in df['word_id'].tolist()]
    print(f"✓ Loaded {len(CLASS_LABELS)} class labels.")
else:
    CLASS_LABELS = [f"Class_{i}" for i in range(157)] # Fallback
    print("⚠ Classes CSV not found. Using fallback labels.")

In [ ]:
class ThreadedCamera:
    """Background daemon thread keeps queue fresh (size 1 = always latest frame)."""
    def __init__(self, src=0):
        self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,  DISPLAY_WIDTH)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, DISPLAY_HEIGHT)
        self.cap.set(cv2.CAP_PROP_FPS, 30)
        self._q      = queue.Queue(maxsize=1)
        self._active = True
        self._thread = threading.Thread(target=self._reader, daemon=True)
        self._thread.start()

    def _reader(self):
        while self._active:
            ret, frame = self.cap.read()
            if not ret:
                continue
            if not self._q.empty():
                try:
                    self._q.get_nowait()
                except queue.Empty:
                    pass
            self._q.put(frame)

    def read(self):
        return self._q.get()

    def is_opened(self):
        return self.cap.isOpened()

    def release(self):
        self._active = False
        self._thread.join(timeout=2)
        self.cap.release()

print("✓ ThreadedCamera defined")

In [ ]:
import mediapipe as mp
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

holistic = mp_holistic.Holistic(
    static_image_mode        = False,
    min_detection_confidence = MP_DETECTION_CONFIDENCE,
    min_tracking_confidence  = MP_TRACKING_CONFIDENCE,
)

def compute_angle(a, b, c):
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.arccos(cosine)

ANGLE_TRIPLETS = [
    (0, 1, 2), (1, 2, 3), (2, 3, 4),
    (0, 5, 6), (5, 6, 7), (6, 7, 8),
    (0, 9, 10), (9, 10, 11), (10, 11, 12),
    (0, 13, 14), (13, 14, 15), (14, 15, 16),
    (0, 17, 18), (17, 18, 19), (18, 19, 20),
]

def extract_engineered_features(landmarks_array):
    if landmarks_array.shape[0] == 0:
        return np.zeros(78, dtype=np.float32)
    wrist = landmarks_array[0]
    relative = landmarks_array - wrist
    relative_flat = relative.flatten()

    angles = []
    for a_idx, b_idx, c_idx in ANGLE_TRIPLETS:
        angle = compute_angle(
            landmarks_array[a_idx],
            landmarks_array[b_idx],
            landmarks_array[c_idx]
        )
        angles.append(angle)
    angles = np.array(angles, dtype=np.float32)
    return np.concatenate([relative_flat, angles])

def extract_features(results):
    # Extract left hand
    if results.left_hand_landmarks:
        lh = np.array([[p.x, p.y, p.z] for p in results.left_hand_landmarks.landmark], dtype=np.float32)
    else:
        lh = np.zeros((21, 3), dtype=np.float32)
        
    # Extract right hand
    if results.right_hand_landmarks:
        rh = np.array([[p.x, p.y, p.z] for p in results.right_hand_landmarks.landmark], dtype=np.float32)
    else:
        rh = np.zeros((21, 3), dtype=np.float32)
    
    # We will use either right hand or left hand (whichever is present, prioritizing right)
    # Alternatively, for words, we can combine both or just track the primary hand.
    # We'll extract features for both and sum or concatenate? 
    # Our updated training script will expect 78-dim (max of left/right) or 156-dim.
    # Since we want to use 78-dim (from the user's instructions "78-dimensional feature engineering"),
    # we will prioritize right hand, if empty, use left hand.
    
    if np.sum(rh) != 0:
        feats = extract_engineered_features(rh)
    elif np.sum(lh) != 0:
        # mirror X coordinates for left hand to act like right hand
        lh[:, 0] = 1.0 - lh[:, 0] 
        feats = extract_engineered_features(lh)
    else:
        feats = np.zeros(78, dtype=np.float32)
        
    return feats

print("✓ MediaPipe Holistic configured with 78-dim engineered features")

In [ ]:
class StabilizationTracker:
    def __init__(self):
        self.buffer          = deque(maxlen=STABILIZATION_WINDOW_SIZE)
        self.committed_label = None
        self.cooldown_until  = 0.0

    def update(self, label, confidence):
        now = time.time()
        if label is None or label == "nothing":
            self.buffer.clear()
            self.committed_label = None
            return None, 0.0, "waiting", 0
            
        if self.committed_label == label and now < self.cooldown_until:
            return label, confidence, "cooldown", 100
            
        if self.committed_label is not None and self.committed_label != label:
            self.committed_label = None   
            
        self.buffer.append(label)
        count    = self.buffer.count(label)
        progress = int(min(100, (count / STABILIZATION_THRESHOLD) * 100))
        
        if count >= STABILIZATION_THRESHOLD and len(self.buffer) == STABILIZATION_WINDOW_SIZE:
            self.committed_label = label
            self.cooldown_until  = now + HOLD_COOLDOWN_SECONDS
            self.buffer.clear()
            return label, confidence, "committed", 100
            
        return label, confidence, "stabilizing", progress

print("✓ StabilizationTracker defined")

In [ ]:
# Note: Custom Layers must be registered if defined in training notebook
class TemporalAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)
    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = tf.reduce_sum(x * a, axis=1)
        return output

model = None
try:
    if os.path.exists(MODEL_PATH):
        model = tf.keras.models.load_model(MODEL_PATH, custom_objects={'TemporalAttention': TemporalAttention})
        print("✓ BiLSTM model loaded")
    else:
        print(f"❌ Model not found at {MODEL_PATH}")
except Exception as e:
    print(f"❌ Error loading model: {e}")

if model is not None:
    @tf.function(input_signature=[tf.TensorSpec(shape=[1, SEQUENCE_LENGTH, 78], dtype=tf.float32)])
    def run_inference(x):
        return model(x, training=False)

    _ = run_inference(tf.zeros([1, SEQUENCE_LENGTH, 78], dtype=tf.float32))
    print("✓ Inference compiled and warmed up")

In [ ]:
def run_sign_recognition():
    if model is None:
        print("❌ Model not loaded.")
        return
        
    cam = ThreadedCamera(src=0)
    if not cam.is_opened():
        print("❌ Cannot access camera")
        return
        
    print("\n================================================")
    print("🤟 WLASL WORD RECOGNITION — PRODUCTION")
    print("================================================")
    print("  q = quit    c = clear sentence")
    print("================================================\n")
    
    window_name = "WLASL SLR — Production"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, DISPLAY_WIDTH, DISPLAY_HEIGHT)
    
    tracker            = StabilizationTracker()
    predicted_sentence = ""
    fps_start          = time.time()
    frame_count        = 0
    fps_display        = 0
    
    # 30-frame sliding window
    sequence_buffer = deque(maxlen=SEQUENCE_LENGTH)
    
    try:
        while True:
            frame = cam.read()
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = holistic.process(rgb)
            rgb.flags.writeable = True
            
            display_status = "Waiting for frames..."
            status_color   = (150, 150, 150)
            
            # Draw landmarks for visualization
            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            elif results.left_hand_landmarks:
                mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
                
            features = extract_features(results)
            sequence_buffer.append(features)
            
            if len(sequence_buffer) == SEQUENCE_LENGTH:
                # Shape: (1, 30, 78)
                model_input = np.expand_dict(np.array(sequence_buffer), axis=0)
                
                # Inference
                prediction = run_inference(tf.constant(model_input))[0].numpy()
                class_idx  = int(np.argmax(prediction))
                conf       = float(prediction[class_idx])
                raw_label  = CLASS_LABELS[class_idx]
                
                if conf < MIN_CONFIDENCE:
                    display_status = f"Low confidence: {conf:.0%}"
                    status_color   = (0, 100, 255)
                    tracker.update(None, 0.0)
                else:
                    tracked_label, tracked_conf, status, progress = tracker.update(raw_label, conf)
                    if status == "stabilizing":
                        display_status = f"{tracked_label} ({tracked_conf:.0%})  {progress}%"
                        status_color   = (0, 255, 255)
                    elif status == "cooldown":
                        display_status = f"{tracked_label} ({tracked_conf:.0%})  ✓ Committed"
                        status_color   = (255, 200, 0)
                    elif status == "committed":
                        display_status = f"{tracked_label} ({tracked_conf:.0%})  ✓ COMMITTED!"
                        status_color   = (0, 255, 0)
                        if tracked_label not in ["nothing", "anonymous"]:
                            predicted_sentence += tracked_label + " "
            else:
                display_status = f"Buffering... {len(sequence_buffer)}/{SEQUENCE_LENGTH}"
                
            frame = cv2.flip(frame, 1)
            frame_count += 1
            if time.time() - fps_start >= 1.0:
                fps_display = frame_count
                frame_count = 0
                fps_start   = time.time()
                
            cv2.rectangle(frame, (0, 0), (DISPLAY_WIDTH, 70), (30, 30, 30), -1)
            cv2.putText(frame, f"FPS: {fps_display}  |  q=quit  c=clear",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            cv2.putText(frame, display_status,
                        (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 2)
                        
            bar_h = 80
            cv2.rectangle(frame,
                          (0, DISPLAY_HEIGHT - bar_h), (DISPLAY_WIDTH, DISPLAY_HEIGHT),
                          (30, 30, 30), -1)
            cv2.putText(frame,
                        predicted_sentence[-40:] if predicted_sentence else "_",
                        (20, DISPLAY_HEIGHT - 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 2)
                        
            cv2.imshow(window_name, frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("c"):
                predicted_sentence = ""
                tracker.update(None, 0.0)
                sequence_buffer.clear()
                print("🗑 Sentence cleared")
                
    finally:
        cam.release()
        cv2.destroyAllWindows()
        print(f"\n📝 Final sentence: {predicted_sentence}")

run_sign_recognition()